# Lab 06: Stretching, Rotating, and Shearing

## Matrices as geometric machines

This lab is a deeper computational companion to Chapter 6. The goal is not only to practice matrix multiplication. The goal is to **see** matrix multiplication.

A $2\times 2$ matrix transforms the entire plane. In this lab you will:

- visualize basis vectors and column vectors,
- transform grids, shapes, and data clouds,
- explore stretches, reflections, rotations, and shears,
- connect determinant to area change,
- study composition and noncommutativity,
- transform a simple digital picture,
- finish with high-dimensional geometric intuition.

Use this notebook as a writing-and-computation lab. Add explanations in markdown cells as you work.

## 1. Setup

Run the cell below first. It imports NumPy and Matplotlib and defines helper functions for drawing axes, vectors, grids, shapes, and data clouds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)


def transform_points(A, points):
    """Apply a 2 by 2 matrix A to an array of 2D points.
    points should have shape (n, 2).
    """
    A = np.asarray(A, dtype=float)
    points = np.asarray(points, dtype=float)
    return points @ A.T


def draw_axes(ax, lim=4, title=None):
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    if title:
        ax.set_title(title)


def plot_vectors(vectors, labels=None, lim=4, title="Vectors"):
    fig, ax = plt.subplots(figsize=(6, 6))
    draw_axes(ax, lim=lim, title=title)
    if labels is None:
        labels = [f"v{i}" for i in range(len(vectors))]
    for v, label in zip(vectors, labels):
        v = np.asarray(v, dtype=float)
        ax.quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy", scale=1)
        ax.text(v[0] + 0.08, v[1] + 0.08, label, fontsize=12)
    plt.show()


def plot_basis_transform(A, lim=4, title="Basis vectors under A"):
    A = np.asarray(A, dtype=float)
    e1 = np.array([1, 0])
    e2 = np.array([0, 1])
    Ae1 = A @ e1
    Ae2 = A @ e2
    plot_vectors([e1, e2, Ae1, Ae2], ["e1", "e2", "A e1", "A e2"], lim=lim, title=title)
    print("A =")
    print(A)
    print("A e1 =", Ae1, "  first column")
    print("A e2 =", Ae2, "  second column")


def plot_grid_transform(A, lim=3, n_lines=13, title="Grid under A"):
    A = np.asarray(A, dtype=float)
    values = np.linspace(-lim, lim, n_lines)
    fig, ax = plt.subplots(figsize=(7, 7))
    draw_axes(ax, lim=lim*2.2, title=title)
    for v in values:
        line_h = np.column_stack([np.linspace(-lim, lim, 250), np.full(250, v)])
        line_v = np.column_stack([np.full(250, v), np.linspace(-lim, lim, 250)])
        Th = transform_points(A, line_h)
        Tv = transform_points(A, line_v)
        ax.plot(Th[:, 0], Th[:, 1], linewidth=0.8, alpha=0.75)
        ax.plot(Tv[:, 0], Tv[:, 1], linewidth=0.8, alpha=0.75)
    plt.show()


def plot_shape_transform(A, points, lim=4, title="Shape under A"):
    A = np.asarray(A, dtype=float)
    points = np.asarray(points, dtype=float)
    T = transform_points(A, points)
    fig, ax = plt.subplots(figsize=(7, 7))
    draw_axes(ax, lim=lim, title=title)
    ax.plot(points[:, 0], points[:, 1], marker="o", label="original")
    ax.plot(T[:, 0], T[:, 1], marker="o", label="transformed")
    ax.legend()
    plt.show()


def unit_square():
    return np.array([[0,0], [1,0], [1,1], [0,1], [0,0]], dtype=float)


def circle_points(n=400):
    t = np.linspace(0, 2*np.pi, n)
    return np.column_stack([np.cos(t), np.sin(t)])

square = unit_square()
circle = circle_points()

## 2. Warm-up: columns are transformed basis vectors

For a matrix

$$
A = \begin{bmatrix} a & b \\ c & d \end{bmatrix},
$$

the first column is $Ae_1$ and the second column is $Ae_2$.

Run the example below, then change the four entries of `A`.

In [ ]:
A = np.array([[2, 1],
              [0, 1]])

plot_basis_transform(A, lim=4, title="Columns as images of basis vectors")

### Student writing task

In your own words, explain why knowing $Ae_1$ and $Ae_2$ is enough to know $Ax$ for every vector $x$.

Write your answer in a markdown cell below.

## 3. Stretching and squeezing

A diagonal matrix

$$
A = \begin{bmatrix} s_x & 0 \\ 0 & s_y \end{bmatrix}
$$

stretches the horizontal and vertical directions separately.

Try several choices of `sx` and `sy`, including negative values.

In [ ]:
sx = 2.5
sy = 0.6
A = np.array([[sx, 0],
              [0, sy]])

plot_basis_transform(A, lim=4, title="Stretching and squeezing basis vectors")
plot_grid_transform(A, title="Stretching and squeezing the grid")
plot_shape_transform(A, square, lim=4, title="Stretching and squeezing the unit square")

### Experiment

Try:

1. `sx = 3`, `sy = 1`
2. `sx = 1`, `sy = 3`
3. `sx = -1`, `sy = 1`
4. `sx = 1`, `sy = -1`
5. `sx = -2`, `sy = 0.5`

For each case, describe the geometry.

## 4. Reflection and orientation

A reflection preserves distances but reverses orientation.

The matrix

$$
\begin{bmatrix}1&0\\0&-1\end{bmatrix}
$$

reflects across the horizontal axis.

In [ ]:
R_x = np.array([[1, 0],
                [0, -1]])

plot_basis_transform(R_x, lim=3, title="Reflection across the horizontal axis")
plot_grid_transform(R_x, lim=2, title="Reflected grid")
plot_shape_transform(R_x, square, lim=3, title="Reflected unit square")
print("determinant:", np.linalg.det(R_x))

### Student task

Create the matrix that reflects across the vertical axis. Then visualize it and compute its determinant.

In [ ]:
# TODO: replace the matrix below
R_y = np.array([[-1, 0],
                [0, 1]])

plot_basis_transform(R_y, lim=3, title="Reflection across the vertical axis")
print("determinant:", np.linalg.det(R_y))

## 5. Shear transformations

A shear slants space. A horizontal shear is

$$
S_h = \begin{bmatrix}1&k\\0&1\end{bmatrix}.
$$

It sends $(x_1,x_2)$ to $(x_1+kx_2,x_2)$. The higher a point is, the more it slides horizontally.

In [ ]:
k = 1.4
S_h = np.array([[1, k],
                [0, 1]])

plot_basis_transform(S_h, lim=4, title="Horizontal shear")
plot_grid_transform(S_h, lim=2, title="Horizontal shear of the grid")
plot_shape_transform(S_h, square, lim=4, title="Horizontal shear of the square")
print("determinant:", np.linalg.det(S_h))

### Student task

A vertical shear has the form

$$
S_v = \begin{bmatrix}1&0\\k&1\end{bmatrix}.
$$

Make a vertical shear with `k = -1.2`. Visualize its effect on a grid and on the unit square.

In [ ]:
# TODO
k = -1.2
S_v = np.array([[1, 0],
                [k, 1]])

plot_grid_transform(S_v, lim=2, title="Vertical shear")
plot_shape_transform(S_v, square, lim=4, title="Vertical shear of the unit square")
print("determinant:", np.linalg.det(S_v))

## 6. Rotation matrices

A counterclockwise rotation by angle $\theta$ is

$$
R_\theta = \begin{bmatrix}
\cos\theta & -\sin\theta\\
\sin\theta & \cos\theta
\end{bmatrix}.
$$

Rotation preserves lengths, angles, areas, and orientation.

In [ ]:
theta = np.pi / 6  # 30 degrees
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

plot_basis_transform(R, lim=3, title="Rotation by 30 degrees")
plot_grid_transform(R, lim=2, title="Rotated grid")
plot_shape_transform(R, square, lim=3, title="Rotated unit square")
print("determinant:", np.linalg.det(R))
print("R.T @ R =")
print(R.T @ R)

### Reflection question

What does the calculation `R.T @ R` tell us about the columns of a rotation matrix?

## 7. The unit circle test

The unit square shows area and columns. The unit circle shows distortion of length and angle.

- Rotation keeps the circle a circle.
- Unequal stretching turns the circle into an ellipse.
- Shear turns the circle into a tilted ellipse.
- Collapse turns the circle into a line segment.

In [ ]:
examples = {
    "rotation": R,
    "stretch": np.array([[2.0, 0], [0, 0.6]]),
    "shear": np.array([[1, 1.2], [0, 1]]),
    "collapse": np.array([[1, 2], [2, 4]])
}

for name, A in examples.items():
    plot_shape_transform(A, circle, lim=4, title=f"Unit circle under {name}")

## 8. Determinant as area scaling

The determinant tells how areas change:

$$
\text{area after transformation} = |\det(A)| \cdot \text{area before transformation}.
$$

The unit square has area $1$, so the area of its transformed parallelogram is $|\det(A)|$.

In [ ]:
def polygon_area(points):
    """Area of a polygon with vertices in order."""
    P = np.asarray(points)
    x = P[:, 0]
    y = P[:, 1]
    return 0.5 * abs(np.dot(x[:-1], y[1:]) - np.dot(y[:-1], x[1:]))

A = np.array([[2, 1],
              [1, 3]])
Tsquare = transform_points(A, square)

plot_shape_transform(A, square, lim=5, title="Area change under A")
print("det(A) =", np.linalg.det(A))
print("area of transformed square =", polygon_area(Tsquare))

### Student task

Create three matrices:

1. one with determinant $2$,
2. one with determinant $-2$,
3. one with determinant $0$.

For each, visualize the unit square and explain what happens to area and orientation.

In [ ]:
# TODO: modify these matrices and explain their geometry
A_pos = np.array([[2, 0], [0, 1]])
A_neg = np.array([[2, 0], [0, -1]])
A_zero = np.array([[1, 2], [2, 4]])

for name, A in [("positive determinant", A_pos), ("negative determinant", A_neg), ("zero determinant", A_zero)]:
    print("\n", name)
    print(A)
    print("determinant:", np.linalg.det(A))
    plot_shape_transform(A, square, lim=5, title=name)

## 9. Composition and order

If you first apply $A$ and then apply $B$, the combined matrix is $BA$.

Order matters. Usually $BA \ne AB$.

In [ ]:
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.array([[1, 1.5],
              [0, 1]])

RS = R @ S  # S acts first, then R
SR = S @ R  # R acts first, then S

print("R @ S =")
print(RS)
print("S @ R =")
print(SR)
print("Are they equal?", np.allclose(RS, SR))

plot_grid_transform(RS, lim=2, title="First shear, then rotate: R @ S")
plot_grid_transform(SR, lim=2, title="First rotate, then shear: S @ R")

### Student writing task

Explain why `R @ S` means the shear happens first and the rotation happens second when the vector is written as `(R @ S) @ x`.

## 10. Transforming a data cloud

A dataset with two features is a cloud of points. A matrix can stretch, rotate, shear, or collapse the cloud.

This is a small preview of why linear algebra matters for data science.

In [ ]:
rng = np.random.default_rng(42)
X = rng.normal(size=(400, 2))

A = np.array([[2.0, 0.8],
              [0.2, 0.5]])
Y = transform_points(A, X)

fig, ax = plt.subplots(figsize=(7, 7))
draw_axes(ax, lim=5, title="A transformed data cloud")
ax.scatter(X[:, 0], X[:, 1], alpha=0.3, label="original")
ax.scatter(Y[:, 0], Y[:, 1], alpha=0.3, label="transformed")
ax.legend()
plt.show()

print("det(A):", np.linalg.det(A))
print("covariance before:\n", np.cov(X.T))
print("covariance after:\n", np.cov(Y.T))

### Exploration

Try a matrix with determinant close to zero. What happens to the data cloud?

In [ ]:
A_near_collapse = np.array([[1, 1.01],
                            [1, 1.00]])
Y = transform_points(A_near_collapse, X)

fig, ax = plt.subplots(figsize=(7, 7))
draw_axes(ax, lim=4, title="Near collapse of a data cloud")
ax.scatter(Y[:, 0], Y[:, 1], alpha=0.35)
plt.show()

print("det(A_near_collapse):", np.linalg.det(A_near_collapse))

## 11. Transforming a small picture

We can represent a drawing as a list of points. Applying a matrix to the points transforms the drawing.

In [ ]:
# A simple house drawing
house = np.array([
    [-1, -1], [1, -1], [1, 0.5], [0, 1.4], [-1, 0.5], [-1, -1],
    [1, 0.5], [-1, 0.5],
    [-0.35, -1], [-0.35, -0.1], [0.35, -0.1], [0.35, -1]
])

A = np.array([[1.2, 0.7],
              [0.1, 1.1]])

plot_shape_transform(A, house, lim=3.5, title="A matrix transforms a drawing")

### Student task

Create three versions of the house:

1. a rotated house,
2. a reflected house,
3. a strongly sheared house.

Then write a short explanation of how each matrix changed the picture.

In [ ]:
# TODO: create your own transformations
A_rotate = np.array([[0, -1], [1, 0]])
A_reflect = np.array([[1, 0], [0, -1]])
A_shear = np.array([[1, 1.8], [0, 1]])

for name, A in [("rotated", A_rotate), ("reflected", A_reflect), ("sheared", A_shear)]:
    plot_shape_transform(A, house, lim=4, title=f"House: {name}")

## 12. High-dimensional extension

In higher dimensions, matrices still transform space. We cannot easily draw $\mathbb R^{50}$, but the same ideas remain:

- columns tell where basis vectors go,
- determinant gives volume scaling for square matrices,
- near-dependent columns create near-collapse,
- transformations can stretch some directions much more than others.

One way to measure how much a matrix stretches vectors is to sample many unit vectors and look at the lengths of their images.

In [ ]:
def random_unit_vectors(n_vectors, dim, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    X = rng.normal(size=(n_vectors, dim))
    return X / np.linalg.norm(X, axis=1, keepdims=True)

rng = np.random.default_rng(123)
dim = 50
n = 5000
U = random_unit_vectors(n, dim, rng)

# A high-dimensional diagonal transformation
scales = np.linspace(0.2, 5.0, dim)
A_high = np.diag(scales)
Y = U @ A_high.T
lengths = np.linalg.norm(Y, axis=1)

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=40)
plt.xlabel("length of A u for random unit vector u")
plt.ylabel("count")
plt.title("Stretching random directions in 50 dimensions")
plt.show()

print("minimum sampled stretch:", lengths.min())
print("maximum sampled stretch:", lengths.max())
print("average sampled stretch:", lengths.mean())

### Final reflection

In two or three paragraphs, answer:

1. Why is a matrix more than a table of numbers?
2. Why are the images of the basis vectors so important?
3. What does determinant tell you geometrically?
4. What changes when we move from 2D to high-dimensional spaces?

## Optional challenge: build your own matrix gallery

Create a function that takes a list of matrices and automatically produces a gallery showing the transformed grid, transformed square, determinant, and a short label for each matrix.

In [ ]:
# Optional challenge starter

def matrix_gallery(matrices, names, lim=2):
    for A, name in zip(matrices, names):
        print("\n" + name)
        print(A)
        print("determinant:", np.linalg.det(A))
        plot_grid_transform(A, lim=lim, title=name)

# Try it
matrices = [
    np.eye(2),
    np.array([[2, 0], [0, 0.5]]),
    np.array([[1, 1.5], [0, 1]]),
    np.array([[0, -1], [1, 0]]),
    np.array([[1, 2], [2, 4]])
]
names = ["identity", "stretch/squeeze", "shear", "rotation", "collapse"]

matrix_gallery(matrices, names)